# 01 - Patient Data Organization

This notebook organizes raw patient data (audio, transcripts, pre-extracted features) into a unified per-patient folder structure.

**Input** (from parent folder):
- `audio/` — WAV/MP3 files per patient
- `transcripts/` — CSV/TXT transcript files
- `features/` — Pre-extracted feature CSVs/MATs

**Output** (`data/patients/{patient_id}/`):
- `audio.wav` — audio file
- `transcript.csv` — transcript
- `audio_features/` — BoAW, OpenSMILE eGeMAPS, MFCC
- `video_features/` — OpenFace, CNN ResNet/VGG/DenseNet

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Ensure the Model src is on path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.organize_patient_data import organize_patient_data, generate_patient_manifest

# Configure paths
SOURCE_DIR = Path("../path_to_parent_folder")  # <-- UPDATE THIS
OUTPUT_DIR = Path("../data/patients")

print(f"Source: {SOURCE_DIR}")
print(f"Output: {OUTPUT_DIR}")

## 1.1 Organize all patient data

In [ ]:
registry = organize_patient_data(
    source_dir=SOURCE_DIR,
    output_dir=OUTPUT_DIR,
    audio_dir_name="audio",
    transcript_dir_name="transcripts",
    features_dir_name="features",
    copy_mode=False,  # True to copy, False to symlink
)

print(f"\nOrganized {len(registry)} patients.")

## 1.2 Generate patient manifest

In [ ]:
manifest = generate_patient_manifest(OUTPUT_DIR)
display(manifest.head(10))

print(f"\nSummary:")
print(f"  Total patients: {len(manifest)}")
print(f"  With audio: {manifest['has_audio'].sum()}")
print(f"  With transcript: {manifest['has_transcript'].sum()}")
print(f"  Avg audio features: {manifest['num_audio_features'].mean():.1f}")
print(f"  Avg video features: {manifest['num_video_features'].mean():.1f}")

## 1.3 Inspect a single patient folder

In [ ]:
import os

sample_pid = manifest.iloc[0]['patient_id']
patient_folder = OUTPUT_DIR / sample_pid

print(f"Patient: {sample_pid}")
for root, dirs, files in os.walk(patient_folder):
    level = root.replace(str(patient_folder), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # limit display
        print(f"{subindent}{file}")
    if len(files) > 5:
        print(f"{subindent}... ({len(files)-5} more files)")